In [4]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pickle
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
try:
    train = pickle.load(open("./windows/train_windows.pkl", "rb"))
    companies = pickle.load(open("./windows/company_list.pkl", "rb"))
    print("✅ Loaded pre-normalized window data.")
except FileNotFoundError:
    print("❌ ERROR: Could not find window files (.pkl) in ./windows/ folder.")
    raise

Using device: cuda
✅ Loaded pre-normalized window data.


In [5]:
import pandas as pd
import numpy as np
import pickle
import random
import os
import glob
from tqdm import tqdm

# --- Configuration ---
INPUT_DIR = "Processed"
OUTPUT_DIR = "windows"
WINDOW_SIZE = 8
TARGET_COLUMN = 'Close'
TEST_SPLIT_RATIO = 0.2 # 10% for test, 90% for train
# ---------------------

# --- 1. SEPARATE FEATURE LISTS ---
financial_features = [
    'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 
    'Daily_Return', 'EMA_7', 'EMA_21'
]
sentiment_features = ['negative', 'neutral', 'positive']

TARGET_COL_INDEX = 1 # 'Close' is 1st in financial_features
NUM_FINANCIAL_FEATURES = len(financial_features)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 2. Get All Processed CSVs ---
all_files = glob.glob(os.path.join(INPUT_DIR, "*.csv"))
if not all_files:
    print(f"❌ ERROR: No '*.csv' files found in '{INPUT_DIR}'.")
    exit()

print(f"Found {len(all_files)} company files to process.")

all_windows = [] 
all_tickers = []

# --- 3. Process Each Company ---
for file in tqdm(all_files, desc="Processing Companies"):
    ticker = os.path.basename(file).split('.')[0]
    all_tickers.append(ticker)
    
    df = pd.read_csv(file)
    df_clean = df.dropna(subset=financial_features)
    
    if len(df_clean) <= WINDOW_SIZE:
        print(f"Skipping {ticker}: not enough data")
        continue
        
    # --- 3b. Create Windows ---
    df[sentiment_features] = df[sentiment_features].fillna(0.0)
    
    features_financial_raw = df_clean[financial_features].values
    features_sentiment_orig = df_clean[sentiment_features].values
    features_combined = np.concatenate([features_financial_raw, features_sentiment_orig], axis=1)
    
    for i in range(len(features_combined) - WINDOW_SIZE):
        window_X = features_combined[i : i + WINDOW_SIZE]
        target_y = features_combined[i + WINDOW_SIZE, TARGET_COL_INDEX]
        
        if not (np.isnan(window_X).any() or np.isnan(target_y)):
            all_windows.append((window_X, target_y, ticker))

# --- 4. Split by Company (to prevent data leakage) ---
print(f"\nCreated {len(all_windows)} total windows from {len(all_tickers)} companies.")

print("Shuffling company list for train/test split...")
random.shuffle(all_tickers)

split_index = int(len(all_tickers) * (1 - TEST_SPLIT_RATIO))
train_tickers = all_tickers[:split_index]
test_tickers = all_tickers[split_index:]

test_ticker_set = set(test_tickers)

print("Splitting windows by company...")
train_windows = []
test_windows = []

# ==================== THIS IS THE FIX ====================
# We must keep the 'ticker' so the training script
# can use it for the embedding layer.
# =======================================================
for window_X, target_y, ticker in all_windows:
    if ticker in test_ticker_set:
        # Before: test_windows.append((window_X, target_y))
        test_windows.append((window_X, target_y, ticker)) # 
    else:
        # Before: train_windows.append((window_X, target_y))
        train_windows.append((window_X, target_y, ticker)) # <-- Keep ticker
# ================= END OF FIX ==========================

random.shuffle(train_windows)
random.shuffle(test_windows)

print(f" > Training companies: {len(train_tickers)} ({len(train_windows)} samples)")
print(f" > Testing companies:  {len(test_tickers)} ({len(test_windows)} samples)")

# --- 5. Save Files ---
print(f"Saving files to '{OUTPUT_DIR}' directory...")
with open(os.path.join(OUTPUT_DIR, "train_windows.pkl"), "wb") as f:
    pickle.dump(train_windows, f)
    
with open(os.path.join(OUTPUT_DIR, "test_windows.pkl"), "wb") as f:
    pickle.dump(test_windows, f)

# We save the full company list, which is correct.
with open(os.path.join(OUTPUT_DIR, "company_list.pkl"), "wb") as f:
    pickle.dump(all_tickers, f)

print("\n" + "="*40)
print("✅ Preprocessing complete!")
print("="*40)

Found 10 company files to process.


Processing Companies: 100%|██████████| 10/10 [00:00<00:00, 60.35it/s]


Created 9910 total windows from 10 companies.
Shuffling company list for train/test split...
Splitting windows by company...
 > Training companies: 8 (7928 samples)
 > Testing companies:  2 (1982 samples)
Saving files to 'windows' directory...



✅ Preprocessing complete!


In [6]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pickle
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import os

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# ------------------------
# Load PRE-NORMALIZED windows
# ------------------------
OUTPUT_DIR = "windows"

try:
    train = pickle.load(open(os.path.join(OUTPUT_DIR, "train_windows.pkl"), "rb"))
    companies = pickle.load(open(os.path.join(OUTPUT_DIR, "company_list.pkl"), "rb"))
    print("✅ Loaded 'train_windows.pkl', 'test_windows.pkl', and 'company_list.pkl'.")
except FileNotFoundError:
    print(f"❌ ERROR: Could not find .pkl files in '{OUTPUT_DIR}' folder.")
    print("Please run the '01_preprocess_data.py' script first.")
    exit()


# ------------------------
# Encode companies
# ------------------------
enc = LabelEncoder().fit(companies)
train = [(X, y, enc.transform([t])[0]) for X, y, t in train]
print("✅ Encoded company tickers to integers.")

# ------------------------
# Dataset and DataLoader
# ------------------------
class WindowDataset(Dataset):
    def __init__(self, data): 
        self.data = data
    def __len__(self): 
        return len(self.data)
    def __getitem__(self, idx):
        X, y, c = self.data[idx]
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        if not np.isfinite(y):
            y = 0.0
        
        return (
            torch.tensor(X, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32),
            torch.tensor(c, dtype=torch.long)
        )

train_loader = DataLoader(WindowDataset(train), batch_size=64, shuffle=True)

# ------------------------
# Transformer model (WITH STABILITY FIX)
# ------------------------
class StockTransformer(nn.Module):
    def __init__(self, feature_dim, num_companies):
        super().__init__()
        d_model = 128
        self.input_proj = nn.Linear(feature_dim, d_model)
        self.company_emb = nn.Embedding(num_companies, d_model)
        
        # --- FIX 1: Add Layer Normalization ---
        self.norm = nn.LayerNorm(d_model) 
        
        enc_layer = nn.TransformerEncoderLayer(d_model, nhead=4, batch_first=True, activation='gelu')
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=2)
        self.head = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, 1))
        
    def forward(self, x, c):
        x = self.input_proj(x)
        x = x + self.company_emb(c).unsqueeze(1) 
        x = self.norm(x)
        x = self.encoder(x)
        return self.head(x[:, -1, :]).squeeze(-1)

feature_dim = train[0][0].shape[1]
model = StockTransformer(feature_dim, len(companies)).to(DEVICE)
print(f"Model created with feature_dim={feature_dim} and num_companies={len(companies)}.")

# ------------------------
# Optimizer & Loss
# ------------------------
opt = torch.optim.Adam(model.parameters(), lr=1e-5)
loss_fn = nn.MSELoss()

# ------------------------
# Training loop
# ------------------------
EPOCHS = 1500 
GRAD_CLIP = 0.5 

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    batches_skipped = 0
    
    for X, y, c in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        X, y, c = X.to(DEVICE), y.to(DEVICE), c.to(DEVICE)
        
        opt.zero_grad()
        pred = model(X, c)
        
        if torch.isnan(pred).any() or torch.isinf(pred).any():
            batches_skipped += 1
            continue
            
        loss = loss_fn(pred, y)
        
        if torch.isnan(loss) or torch.isinf(loss):
            batches_skipped += 1
            continue
            
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        opt.step()
        
        total_loss += loss.item()
    
    if batches_skipped > 0:
        print(f"⚠️ Skipped {batches_skipped} batches due to NaN/Inf.")
    
    num_valid_batches = len(train_loader) - batches_skipped
    avg_train_loss = total_loss / (num_valid_batches + 1e-6)
    print(f"Epoch {epoch+1} Train Loss: {avg_train_loss:.6f}")
# ------------------------
# Save model & data
# ------------------------
torch.save({
    "model_state": model.state_dict(),
    "companies": companies
}, "./model_data.pt")

print("✅ Model and metadata saved as model_data.pt")

Using device: cuda
✅ Loaded 'train_windows.pkl', 'test_windows.pkl', and 'company_list.pkl'.
✅ Encoded company tickers to integers.
Model created with feature_dim=12 and num_companies=10.


Epoch 1/1500: 100%|██████████| 124/124 [00:01<00:00, 80.67it/s]


Epoch 1 Train Loss: 33004.322425


Epoch 2/1500: 100%|██████████| 124/124 [00:01<00:00, 70.71it/s]


Epoch 2 Train Loss: 32597.990964


...

Epoch 1498 Train Loss: 6132.391660


Epoch 1499/1500: 100%|██████████| 124/124 [00:01<00:00, 74.03it/s]


Epoch 1499 Train Loss: 6244.174731


Epoch 1500/1500: 100%|██████████| 124/124 [00:01<00:00, 69.87it/s]

Epoch 1500 Train Loss: 5741.894544
✅ Model and metadata saved as model_data.pt
